# TrexQuant: 10-Experiment Mega-Matrix Pipeline

**Metric:** Pearson Correlation Coefficient (IC)  
**Competition:** Earnings Return Prediction Challenge 2025 Q4  

## Pipeline Overview
| Phase | Experiments | Theme |
|-------|-------------|-------|
| 1 | Exp 1 | Masked Denoising Autoencoder (DAE) — unsupervised regime fix |
| 2 | Exp 2-3 | LightGBM Mega-Matrix baselines |
| 3 | Exp 4-5 | Target engineering (rank / robust-z) |
| 4 | Exp 6-8 | Model diversity: CatBoost, XGBoost, Cross-sectional MLP |
| 5 | Exp 9-10 | Ensembling: equal-weight blend → SLSQP optimal blend + submission |


## 0. Global Config & Imports

In [ ]:
import os, gc, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from scipy.sparse import hstack as sp_hstack, csr_matrix
warnings.filterwarnings("ignore")

# ── seeds ──────────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── mode ───────────────────────────────────────────────────────────────────
# Set FAST_MODE=True for a quick smoke-test (fewer trees / epochs)
FAST_MODE = False

# ── paths ──────────────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

# ── global experiment tracker ──────────────────────────────────────────────
results: dict = {}   # {exp_name: {"ic": float|None, "note": str}}

# ── OOF / pred collectors (filled progressively) ──────────────────────────
oof_preds:  dict = {}   # {model_key: np.ndarray shape (n_train,)}
test_preds: dict = {}   # {model_key: np.ndarray shape (n_test,)}

# ── timing helper ─────────────────────────────────────────────────────────
_timers: dict = {}
def tic(tag): _timers[tag] = time.time()
def toc(tag):
    t = time.time() - _timers.get(tag, time.time())
    print(f"  [{tag}] done in {t:.1f}s")

print("Config OK  |  FAST_MODE =", FAST_MODE)


## 1. Utility — Robust Pearson IC Helper

In [ ]:
def pearson_ic(a: np.ndarray, b: np.ndarray) -> float:
    """Safe Pearson r; returns NaN on degenerate inputs."""
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return float("nan")
    aa = a[mask] - a[mask].mean()
    bb = b[mask] - b[mask].mean()
    denom = np.sqrt((aa**2).sum()) * np.sqrt((bb**2).sum())
    return float((aa * bb).sum() / denom) if denom > 1e-12 else float("nan")


def per_date_ic(oof: np.ndarray, y: np.ndarray, dates: np.ndarray) -> float:
    """Mean Pearson IC averaged across unique dates."""
    ics = []
    for d in np.unique(dates):
        m = dates == d
        if m.sum() < 3:
            continue
        ic = pearson_ic(oof[m], y[m])
        if np.isfinite(ic):
            ics.append(ic)
    return float(np.mean(ics)) if ics else float("nan")


def log_result(name: str, ic, note=""):
    ic_val = float("nan") if ic is None else float(ic)
    ic_rounded = round(ic_val, 6) if np.isfinite(ic_val) else None
    results[name] = {"ic": ic_rounded, "note": note}
    print(f"  >>> {name}: OOF IC = {ic}  | {note}")


print("Pearson helpers loaded.")


## 2. Data Loading & Column Detection

In [ ]:
tic("data_load")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
toc("data_load")
print(f"train: {train.shape}  |  test: {test.shape}")

# ── column detection ──────────────────────────────────────────────────────
ID_COL      = "id"
TARGET_COL  = "target"
DATE_COL    = "di"
ENTITY_COL  = "si"
SECTOR_COL  = "sector"   if "sector"   in train.columns else None
INDUSTRY_COL= "industry" if "industry" in train.columns else None

LIQ_COLS = [c for c in ["top500","top1000","top2000"] if c in train.columns]

META_COLS = [c for c in [DATE_COL, ENTITY_COL, SECTOR_COL, INDUSTRY_COL] + LIQ_COLS if c is not None]

# Feature columns: f_ prefix (or plain f followed by digits)
f_cols = [c for c in train.columns
          if (c.startswith("f_") or (c.startswith("f") and c[1:].replace("_","").isdigit()))
          and c not in META_COLS]

# Drop constant / all-nan feature columns in train
valid_f = []
for c in f_cols:
    col = train[c]
    if col.nunique(dropna=False) > 1 and col.notna().any():
        valid_f.append(c)
f_cols = valid_f

N_FEATS = len(f_cols)
print(f"Features detected: {N_FEATS}")
print(f"Meta cols: {META_COLS}")
print(f"Sector: {SECTOR_COL}  |  Industry: {INDUSTRY_COL}")
print(f"Liquidity flags: {LIQ_COLS}")


## 3. Preprocessing — Imputation & Scaling

In [ ]:
# ── fill NaN with feature median (computed on train only) ─────────────────
f_medians = train[f_cols].median()
train[f_cols] = train[f_cols].fillna(f_medians)
test[f_cols]  = test[f_cols].fillna(f_medians)

# ── clip extreme values to [-10σ, +10σ] ───────────────────────────────────
f_mean = train[f_cols].mean()
f_std  = train[f_cols].std().replace(0, 1)
train[f_cols] = ((train[f_cols] - f_mean) / f_std).clip(-10, 10)
test[f_cols]  = ((test[f_cols]  - f_mean) / f_std).clip(-10, 10)

y_train = train[TARGET_COL].values.astype(np.float32)
di_train = train[DATE_COL].values
di_test  = test[DATE_COL].values

X_raw_tr = train[f_cols].values.astype(np.float32)
X_raw_te = test[f_cols].values.astype(np.float32)

print(f"X_raw_tr: {X_raw_tr.shape}  |  X_raw_te: {X_raw_te.shape}")
print(f"y_train finite: {np.isfinite(y_train).sum()} / {len(y_train)}")


## 4. Time-Series Walk-Forward CV Folds

In [ ]:
# Purged walk-forward: 5 folds by date, embargo = 1 period
unique_dates = np.sort(np.unique(di_train))
N_DATES = len(unique_dates)
N_FOLDS = 5
EMBARGO  = 1   # number of date periods to skip at fold boundary

fold_size = N_DATES // N_FOLDS
FOLDS = []

for fold in range(N_FOLDS):
    val_start_idx = (fold + 1) * fold_size  if fold < N_FOLDS - 1 else N_DATES
    # Use all history up to val_start as train, then val window
    train_end_idx = fold * fold_size + fold_size
    if fold == 0:
        # first fold: need at least some train data
        train_cutoff = N_DATES // (N_FOLDS + 1)
    else:
        train_cutoff = fold * fold_size

    val_end_idx   = min(train_cutoff + 2 * fold_size, N_DATES)
    val_start_idx = train_cutoff + EMBARGO

    train_dates = unique_dates[:train_cutoff]
    val_dates   = unique_dates[val_start_idx:val_end_idx]

    if len(train_dates) == 0 or len(val_dates) == 0:
        continue

    tr_idx  = np.where(np.isin(di_train, train_dates))[0]
    val_idx = np.where(np.isin(di_train, val_dates))[0]
    FOLDS.append((tr_idx, val_idx))

print(f"Walk-forward folds created: {len(FOLDS)}")
for i, (tr, va) in enumerate(FOLDS):
    print(f"  Fold {i+1}: train={len(tr):>7,}  val={len(va):>7,}")


---
## PHASE 1 — Unsupervised Representation Learning

**Hypothesis:** A denoising autoencoder trained on the joint train+test feature space
learns a compact 32-d manifold that is robust to regime drift between training and test
periods. These embeddings act as macro-context signals for downstream tree models.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")


### Experiment 1: Masked Denoising Autoencoder (DAE)

In [ ]:
class MaskedAutoencoder(nn.Module):
    def __init__(self, n_feats: int, hidden: int = 128, bottleneck: int = 32, dropout: float = 0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_feats, hidden),
            nn.Mish(),
            nn.Dropout(dropout),
            nn.Linear(hidden, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, hidden),
            nn.Mish(),
            nn.Linear(hidden, n_feats),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


# ── build training set: train + test features concatenated ────────────────
X_all = np.vstack([X_raw_tr, X_raw_te]).astype(np.float32)

MASK_RATE = 0.25
DAE_EPOCHS = 5 if FAST_MODE else 10
DAE_BATCH  = 2048
DAE_LR     = 1e-3

torch.manual_seed(SEED)
dae = MaskedAutoencoder(N_FEATS).to(DEVICE)
optimizer_dae = optim.Adam(dae.parameters(), lr=DAE_LR)
criterion_dae = nn.MSELoss()

ds_all = TensorDataset(torch.from_numpy(X_all))
loader_all = DataLoader(ds_all, batch_size=DAE_BATCH, shuffle=True, drop_last=False)

tic("exp1_dae")
dae.train()
for epoch in range(1, DAE_EPOCHS + 1):
    epoch_loss = 0.0
    for (xb,) in loader_all:
        xb = xb.to(DEVICE)
        # random mask
        mask = (torch.rand_like(xb) < MASK_RATE).float()
        xb_masked = xb * (1.0 - mask)

        optimizer_dae.zero_grad()
        recon, _ = dae(xb_masked)
        loss = criterion_dae(recon, xb)
        loss.backward()
        optimizer_dae.step()
        epoch_loss += loss.item() * len(xb)

    epoch_loss /= len(X_all)
    print(f"  DAE Epoch {epoch}/{DAE_EPOCHS}  loss={epoch_loss:.6f}")

toc("exp1_dae")

# ── extract frozen embeddings ─────────────────────────────────────────────
dae.eval()
with torch.no_grad():
    X_emb_tr = dae.encoder(
        torch.from_numpy(X_raw_tr).to(DEVICE)
    ).cpu().numpy().astype(np.float32)
    X_emb_te = dae.encoder(
        torch.from_numpy(X_raw_te).to(DEVICE)
    ).cpu().numpy().astype(np.float32)

log_result("Exp1_DAE", None, f"Reconstruction IC not applicable; emb shape={X_emb_tr.shape}")

del ds_all, loader_all
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()
print(f"X_emb_tr: {X_emb_tr.shape}  |  X_emb_te: {X_emb_te.shape}")


---
## PHASE 2 — Mega-Matrix Tree Baselines

**Hypothesis:** Concatenating DAE embeddings with raw features gives LightGBM macro-regime
context that prevents overfitting to training-period noise. Beating the raw-only baseline
here validates the DAE approach.


In [ ]:
# Build mega-matrices
X_mega_tr = np.hstack([X_raw_tr, X_emb_tr]).astype(np.float32)
X_mega_te = np.hstack([X_raw_te, X_emb_te]).astype(np.float32)
print(f"X_mega_tr: {X_mega_tr.shape}")


In [ ]:
import lightgbm as lgb
print(f"LightGBM version: {lgb.__version__}")

LGBM_PARAMS_BASE = dict(
    objective      = "regression",
    metric         = "rmse",
    n_estimators   = 200 if FAST_MODE else 600,
    learning_rate  = 0.05,
    num_leaves     = 31,
    min_child_samples = 20,
    subsample      = 0.8,
    colsample_bytree = 0.8,
    reg_alpha      = 0.1,
    reg_lambda     = 1.0,
    random_state   = SEED,
    n_jobs         = -1,
    verbosity      = -1,
)

def run_lgbm_oof(X_tr, X_te, y_tr, folds, params, y_eval=None):
    """Train LightGBM with walk-forward OOF, return (oof_preds, test_preds, mean_ic)."""
    oof  = np.full(len(y_tr), np.nan, dtype=np.float32)
    preds = np.zeros(len(X_te), dtype=np.float32)
    y_eval = y_eval if y_eval is not None else y_tr

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        Xf, yf = X_tr[tr_idx], y_tr[tr_idx]
        Xv     = X_tr[val_idx]
        valid  = np.isfinite(yf)

        model = lgb.LGBMRegressor(**params)
        model.fit(Xf[valid], yf[valid])

        oof[val_idx] = model.predict(Xv)
        preds        += model.predict(X_te) / len(folds)

        fold_ic = pearson_ic(oof[val_idx], y_eval[val_idx])
        print(f"    Fold {fold_i+1}/{len(folds)}  IC={fold_ic:.6f}")

        del model
        gc.collect()

    global_ic = pearson_ic(oof[np.isfinite(oof)], y_eval[np.isfinite(oof)])
    return oof, preds, global_ic


### Experiment 2: LightGBM on Raw + Embeddings (Mega-Matrix)

In [ ]:
tic("exp2_lgbm_mega")
oof2, preds2, ic2 = run_lgbm_oof(X_mega_tr, X_mega_te, y_train, FOLDS, LGBM_PARAMS_BASE)
toc("exp2_lgbm_mega")
log_result("Exp2_LGBM_Mega", ic2, "LightGBM | raw+emb features | raw target")
oof_preds["Exp2"]  = oof2
test_preds["Exp2"] = preds2
gc.collect()


## 5. Feature Engineering — Cross-Sectional Transforms

In [ ]:
# Cross-sectional rank and robust-z features by date (di)
# These are computed strictly per date to avoid look-ahead leakage.
tic("fe")

# Select top features by variance to keep matrix tractable
TOP_K_FE = min(50, N_FEATS)   # top-K features for expensive per-date transforms
feat_var  = train[f_cols].var().sort_values(ascending=False)
top_feats = feat_var.index[:TOP_K_FE].tolist()

def cs_rank_z_features(df, feats, di_col="di"):
    """Add per-date rank and robust-z for selected features."""
    rows = []
    for feat in feats:
        g = df.groupby(di_col)[feat]
        # rank in [0,1]
        rk = g.rank(method="average", pct=True).astype(np.float32)
        # robust z-score
        med = g.transform("median")
        mad = (df[feat] - med).abs().groupby(df[di_col]).transform("median")
        rz  = ((df[feat] - med) / (1.4826 * mad.replace(0, np.nan))).clip(-8, 8).astype(np.float32)
        rows.append((f"{feat}_rk", rk))
        rows.append((f"{feat}_rz", rz))
    return pd.DataFrame({k: v for k, v in rows}, index=df.index)

fe_tr_df = cs_rank_z_features(train, top_feats)
fe_te_df = cs_rank_z_features(test,  top_feats)

# fill any remaining NaN from low-count dates
fe_tr_df = fe_tr_df.fillna(0.0)
fe_te_df = fe_te_df.fillna(0.0)

X_fe_tr = np.hstack([X_raw_tr, fe_tr_df.values]).astype(np.float32)
X_fe_te = np.hstack([X_raw_te, fe_te_df.values]).astype(np.float32)

del fe_tr_df, fe_te_df
gc.collect()

toc("fe")
print(f"X_fe_tr: {X_fe_tr.shape}  |  X_fe_te: {X_fe_te.shape}")


In [ ]:
# Mega-matrix with FE features + DAE embeddings
X_mega_fe_tr = np.hstack([X_fe_tr, X_emb_tr]).astype(np.float32)
X_mega_fe_te = np.hstack([X_fe_te, X_emb_te]).astype(np.float32)
print(f"X_mega_fe_tr: {X_mega_fe_tr.shape}  |  X_mega_fe_te: {X_mega_fe_te.shape}")


### Experiment 3: LightGBM on FE + Embeddings (Mega-FE-Matrix)

In [ ]:
tic("exp3_lgbm_mega_fe")
oof3, preds3, ic3 = run_lgbm_oof(X_mega_fe_tr, X_mega_fe_te, y_train, FOLDS, LGBM_PARAMS_BASE)
toc("exp3_lgbm_mega_fe")
log_result("Exp3_LGBM_MegaFE", ic3, "LightGBM | fe+emb features | raw target")
oof_preds["Exp3"]  = oof3
test_preds["Exp3"] = preds3
gc.collect()


---
## PHASE 3 — Target Engineering

**Hypothesis:** Raw return targets have heavy tails and inter-day heteroskedasticity.
Training on cross-sectionally normalized targets stabilises gradient updates and aligns
the training objective with the Pearson IC evaluation metric.


In [ ]:
def make_rank_target(df, di_col="di"):
    """Per-date percentile rank target in [0, 1]."""
    return df.groupby(di_col)[TARGET_COL].rank(pct=True).values.astype(np.float32)


def make_robustz_target(df, y, di_col="di"):
    """Per-date robust z-score target using median/MAD."""
    s = pd.Series(y, index=df.index, name="__y")
    tmp = df[[di_col]].copy()
    tmp["__y"] = s.values
    med = tmp.groupby(di_col)["__y"].transform("median")
    mad = (tmp["__y"] - med).abs().groupby(tmp[di_col]).transform("median")
    rz  = ((tmp["__y"] - med) / (1.4826 * mad.replace(0, np.nan))).clip(-8, 8)
    return rz.fillna(0.0).values.astype(np.float32)


y_rank    = make_rank_target(train)
y_robustz = make_robustz_target(train, y_train)

print(f"y_rank    | mean={y_rank.mean():.4f}  std={y_rank.std():.4f}")
print(f"y_robustz | mean={y_robustz.mean():.4f}  std={y_robustz.std():.4f}")


### Experiment 4: Mega-Matrix with Cross-Sectional Rank Target

In [ ]:
tic("exp4_rank_target")
oof4, preds4, _ = run_lgbm_oof(
    X_mega_fe_tr, X_mega_fe_te,
    y_rank,             # train on rank
    FOLDS,
    LGBM_PARAMS_BASE,
    y_eval=y_train,     # evaluate IC against raw target
)
toc("exp4_rank_target")

ic4 = pearson_ic(oof4[np.isfinite(oof4)], y_train[np.isfinite(oof4)])
log_result("Exp4_RankTarget", ic4, "LightGBM | fe+emb | rank target → raw IC")
oof_preds["Exp4"]  = oof4
test_preds["Exp4"] = preds4
gc.collect()


### Experiment 5: Mega-Matrix with Robust Z-Score Target

In [ ]:
tic("exp5_robustz_target")
oof5, preds5, _ = run_lgbm_oof(
    X_mega_fe_tr, X_mega_fe_te,
    y_robustz,
    FOLDS,
    LGBM_PARAMS_BASE,
    y_eval=y_train,
)
toc("exp5_robustz_target")

ic5 = pearson_ic(oof5[np.isfinite(oof5)], y_train[np.isfinite(oof5)])
log_result("Exp5_RobustZ", ic5, "LightGBM | fe+emb | robust-z target → raw IC")
oof_preds["Exp5"]  = oof5
test_preds["Exp5"] = preds5

# ── select best target for Phase 4 ────────────────────────────────────────
if ic4 >= ic5:
    y_best  = y_rank
    BEST_TARGET_NAME = "rank"
else:
    y_best  = y_robustz
    BEST_TARGET_NAME = "robust-z"
print(f"Best target for Phase 4: {BEST_TARGET_NAME}  (Exp4 IC={ic4:.6f}, Exp5 IC={ic5:.6f})")
gc.collect()


---
## PHASE 4 — Model Diversity

**Hypothesis:** Predictions from models with different inductive biases (CatBoost ordered
boosting, XGBoost histogram, PyTorch MLP with Pearson loss) will have low pairwise
correlation, making their ensemble meaningfully stronger than any single model.


### Experiment 6: CatBoost on Mega-Matrix

In [ ]:
try:
    from catboost import CatBoostRegressor
    print("CatBoost available.")
    _HAS_CATBOOST = True
except ImportError:
    print("CatBoost not installed — Exp 6 will be skipped.")
    _HAS_CATBOOST = False

if _HAS_CATBOOST:
    tic("exp6_catboost")

    # Build matrix that includes sector/industry as integers for CatBoost
    cat_feature_indices = []
    cat_extra_cols = []
    for col in [SECTOR_COL, INDUSTRY_COL]:
        if col is not None:
            enc_tr = train[col].astype("category").cat.codes.values.reshape(-1, 1)
            enc_te = test[col].astype("category").cat.codes.values.reshape(-1, 1)
            cat_extra_cols.append((enc_tr, enc_te))

    if cat_extra_cols:
        cat_tr_cols = np.hstack([c[0] for c in cat_extra_cols]).astype(np.float32)
        cat_te_cols = np.hstack([c[1] for c in cat_extra_cols]).astype(np.float32)
        X_cb_tr = np.hstack([X_mega_fe_tr, cat_tr_cols])
        X_cb_te = np.hstack([X_mega_fe_te, cat_te_cols])
        cat_feature_indices = list(range(X_mega_fe_tr.shape[1], X_cb_tr.shape[1]))
    else:
        X_cb_tr, X_cb_te = X_mega_fe_tr, X_mega_fe_te

    oof6  = np.full(len(y_train), np.nan, dtype=np.float32)
    preds6 = np.zeros(len(test), dtype=np.float32)

    cb_params = dict(
        iterations   = 200 if FAST_MODE else 500,
        learning_rate= 0.05,
        depth        = 6,
        l2_leaf_reg  = 3.0,
        random_seed  = SEED,
        verbose      = 0,
        loss_function= "RMSE",
        cat_features = cat_feature_indices if cat_feature_indices else None,
    )

    for fold_i, (tr_idx, val_idx) in enumerate(FOLDS):
        Xf, yf = X_cb_tr[tr_idx], y_best[tr_idx]
        Xv     = X_cb_tr[val_idx]
        valid  = np.isfinite(yf)

        if cat_feature_indices:
            cb = CatBoostRegressor(**{k: v for k, v in cb_params.items() if v is not None})
        else:
            cb_p2 = {k: v for k, v in cb_params.items() if k != "cat_features"}
            cb = CatBoostRegressor(**cb_p2)

        cb.fit(Xf[valid], yf[valid])
        oof6[val_idx] = cb.predict(Xv)
        preds6       += cb.predict(X_cb_te) / len(FOLDS)

        fold_ic = pearson_ic(oof6[val_idx], y_train[val_idx])
        print(f"    Fold {fold_i+1}/{len(FOLDS)}  IC={fold_ic:.6f}")
        del cb; gc.collect()

    toc("exp6_catboost")
    ic6 = pearson_ic(oof6[np.isfinite(oof6)], y_train[np.isfinite(oof6)])
    log_result("Exp6_CatBoost", ic6, f"CatBoost | {BEST_TARGET_NAME} target")
    oof_preds["Exp6"]  = oof6
    test_preds["Exp6"] = preds6
else:
    log_result("Exp6_CatBoost", float("nan"), "skipped — catboost not installed")

gc.collect()


### Experiment 7: XGBoost on Mega-Matrix (compatibility-safe)

In [ ]:
try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
    _HAS_XGB = True
except ImportError:
    print("XGBoost not installed — Exp 7 will be skipped.")
    _HAS_XGB = False

if _HAS_XGB:
    tic("exp7_xgb")

    # NOTE: no staged_predict, no early_stopping_rounds in .fit — compatibility-safe
    xgb_params = dict(
        n_estimators    = 300 if FAST_MODE else 600,
        max_depth       = 6,
        learning_rate   = 0.03,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        reg_alpha       = 0.1,
        reg_lambda      = 1.0,
        tree_method     = "hist",
        random_state    = SEED,
        n_jobs          = -1,
        verbosity       = 0,
    )

    oof7  = np.full(len(y_train), np.nan, dtype=np.float32)
    preds7 = np.zeros(len(test), dtype=np.float32)

    for fold_i, (tr_idx, val_idx) in enumerate(FOLDS):
        Xf, yf = X_mega_fe_tr[tr_idx], y_best[tr_idx]
        Xv     = X_mega_fe_tr[val_idx]
        valid  = np.isfinite(yf)

        model = xgb.XGBRegressor(**xgb_params)
        model.fit(Xf[valid], yf[valid], verbose=False)

        oof7[val_idx] = model.predict(Xv)
        preds7       += model.predict(X_mega_fe_te) / len(FOLDS)

        fold_ic = pearson_ic(oof7[val_idx], y_train[val_idx])
        print(f"    Fold {fold_i+1}/{len(FOLDS)}  IC={fold_ic:.6f}")
        del model; gc.collect()

    toc("exp7_xgb")
    ic7 = pearson_ic(oof7[np.isfinite(oof7)], y_train[np.isfinite(oof7)])
    log_result("Exp7_XGBoost", ic7, f"XGBoost hist | {BEST_TARGET_NAME} target")
    oof_preds["Exp7"]  = oof7
    test_preds["Exp7"] = preds7
else:
    log_result("Exp7_XGBoost", float("nan"), "skipped — xgboost not installed")

gc.collect()


### Experiment 8: Cross-Sectional MLP on Embeddings Only

In [ ]:
class PearsonLoss(nn.Module):
    """Differentiable 1 - Pearson correlation loss."""
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred   = pred   - pred.mean()
        target = target - target.mean()
        corr   = (pred * target).sum() / (
            pred.norm() * target.norm() + 1e-8
        )
        return 1.0 - corr


class CrossSectionalMLP(nn.Module):
    def __init__(self, in_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.BatchNorm1d(64),
            nn.Mish(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.Mish(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


MLP_EPOCHS = 10 if FAST_MODE else 30
MLP_LR     = 1e-3
MLP_BATCH  = 512

oof8  = np.full(len(y_train), np.nan, dtype=np.float32)
preds8 = np.zeros(len(test), dtype=np.float32)
pearson_loss_fn = PearsonLoss()

tic("exp8_mlp")
for fold_i, (tr_idx, val_idx) in enumerate(FOLDS):
    Xf  = torch.from_numpy(X_emb_tr[tr_idx]).to(DEVICE)
    yf  = torch.from_numpy(y_train[tr_idx]).to(DEVICE)
    Xv  = torch.from_numpy(X_emb_tr[val_idx]).to(DEVICE)

    valid_mask = torch.isfinite(yf)
    Xf, yf = Xf[valid_mask], yf[valid_mask]

    torch.manual_seed(SEED + fold_i)
    mlp = CrossSectionalMLP(in_dim=X_emb_tr.shape[1]).to(DEVICE)
    opt = optim.Adam(mlp.parameters(), lr=MLP_LR)

    ds_fold = TensorDataset(Xf, yf)
    loader_fold = DataLoader(ds_fold, batch_size=MLP_BATCH, shuffle=True, drop_last=True)

    mlp.train()
    for ep in range(MLP_EPOCHS):
        ep_loss = 0.0
        for xb, yb in loader_fold:
            opt.zero_grad()
            pred_b = mlp(xb)
            loss   = pearson_loss_fn(pred_b, yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item()

    mlp.eval()
    with torch.no_grad():
        oof8[val_idx] = mlp(Xv).cpu().numpy()
        # test preds: predict in batches for memory safety
        Xte_t = torch.from_numpy(X_emb_te).to(DEVICE)
        te_out = []
        for i in range(0, len(Xte_t), 4096):
            te_out.append(mlp(Xte_t[i:i+4096]).cpu().numpy())
        preds8 += np.concatenate(te_out) / len(FOLDS)

    fold_ic = pearson_ic(oof8[val_idx], y_train[val_idx])
    print(f"    Fold {fold_i+1}/{len(FOLDS)}  IC={fold_ic:.6f}")

    del mlp, opt, ds_fold, loader_fold, Xf, yf, Xv
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    gc.collect()

toc("exp8_mlp")
ic8 = pearson_ic(oof8[np.isfinite(oof8)], y_train[np.isfinite(oof8)])
log_result("Exp8_MLP", ic8, "Cross-sectional MLP | 32-d embeddings | Pearson loss")
oof_preds["Exp8"]  = oof8
test_preds["Exp8"] = preds8

if DEVICE.type == "cuda": torch.cuda.empty_cache()
gc.collect()


---
## PHASE 5 — Meta-Ensembling & Post-Processing

**Hypothesis:** Blending models with diverse inductive biases reduces variance while
preserving the signal from each. Optimising blend weights directly on OOF Pearson IC
via SLSQP gives a mathematically tighter upper bound on test performance than equal weighting.


In [ ]:
# Collect the models whose OOF preds are finite and non-trivial
BLEND_KEYS = []
for k in ["Exp5", "Exp6", "Exp7", "Exp8"]:
    if k in oof_preds:
        oo = oof_preds[k]
        fin = np.isfinite(oo)
        if fin.sum() > 100:
            BLEND_KEYS.append(k)
            print(f"  {k}: {fin.sum():,} finite OOF preds  IC={pearson_ic(oo[fin], y_train[fin]):.6f}")
        else:
            print(f"  {k}: skipped (too few finite preds)")
    else:
        print(f"  {k}: not found in oof_preds — skipped")

print(f"
Blend pool: {BLEND_KEYS}")


### Experiment 9: Equal-Weight Blend

In [ ]:
if BLEND_KEYS:
    oof_stack  = np.vstack([oof_preds[k]  for k in BLEND_KEYS]).T   # (n_train, n_models)
    test_stack = np.vstack([test_preds[k] for k in BLEND_KEYS]).T   # (n_test, n_models)

    # Replace NaN with column means before blending
    for col_i in range(oof_stack.shape[1]):
        col = oof_stack[:, col_i]
        col[~np.isfinite(col)] = np.nanmean(col)

    oof9  = oof_stack.mean(axis=1)
    preds9 = test_stack.mean(axis=1)

    ic9 = pearson_ic(oof9, y_train)
    log_result("Exp9_EqualBlend", ic9, f"equal blend of {BLEND_KEYS}")
    oof_preds["Exp9"]  = oof9
    test_preds["Exp9"] = preds9
else:
    print("No blend keys available — falling back to Exp5 or Exp3.")
    fallback = "Exp5" if "Exp5" in oof_preds else "Exp3"
    oof9  = oof_preds[fallback]
    preds9 = test_preds[fallback]
    ic9 = pearson_ic(oof9[np.isfinite(oof9)], y_train[np.isfinite(oof9)])
    log_result("Exp9_EqualBlend", ic9, f"fallback to {fallback}")

gc.collect()


### Experiment 10: Constrained SLSQP Blend + Submission

In [ ]:
if BLEND_KEYS and len(BLEND_KEYS) > 1:
    # Only use rows where y_train is finite
    valid_rows = np.isfinite(y_train)
    O = oof_stack[valid_rows]   # (n_valid, n_models)
    Y = y_train[valid_rows]

    def neg_pearson(w):
        blend = O @ w
        return -pearson_ic(blend, Y)

    n_models = O.shape[1]
    w0 = np.ones(n_models) / n_models
    constraints = {"type": "eq", "fun": lambda w: w.sum() - 1.0}
    bounds = [(0.0, 1.0)] * n_models

    opt_result = minimize(
        neg_pearson, w0,
        method     = "SLSQP",
        bounds     = bounds,
        constraints= constraints,
        options    = {"maxiter": 1000, "ftol": 1e-9},
    )

    w_opt = np.clip(opt_result.x, 0, 1)
    w_opt /= w_opt.sum()
    print(f"Optimal weights: { {k: round(w, 4) for k, w in zip(BLEND_KEYS, w_opt)} }")

    oof10  = oof_stack @ w_opt
    preds10 = test_stack @ w_opt
else:
    print("Single-model pool or empty — using equal weights.")
    oof10   = oof9.copy()
    preds10 = preds9.copy()

ic10 = pearson_ic(oof10[np.isfinite(oof10)], y_train[np.isfinite(oof10)])
log_result("Exp10_SLSQPBlend", ic10, "SLSQP-optimised non-negative blend")

oof_preds["Exp10"]  = oof10
test_preds["Exp10"] = preds10
gc.collect()


### Post-processing: Per-date Z-score Standardisation → submission.csv

In [ ]:
# ── per-date z-score standardisation of test predictions ─────────────────
final_test_preds = preds10.copy()

test_df_pred = test[[DATE_COL, ID_COL]].copy()
test_df_pred["raw_pred"] = final_test_preds

def per_date_zscore(df, pred_col="raw_pred", di_col="di"):
    mu  = df.groupby(di_col)[pred_col].transform("mean")
    sig = df.groupby(di_col)[pred_col].transform("std").replace(0, 1)
    return ((df[pred_col] - mu) / sig).values

test_df_pred["final_pred"] = per_date_zscore(test_df_pred)

# ── sanity checks ─────────────────────────────────────────────────────────
assert test_df_pred["final_pred"].isna().sum() == 0, "NaN in final preds!"
assert (test_df_pred["final_pred"] == 0).mean() < 0.5, "More than 50% zero preds — suspicious!"

submission = test_df_pred[[ID_COL, "final_pred"]].rename(columns={"final_pred": "target"})
submission.to_csv("submission.csv", index=False)

print(f"submission.csv saved: {len(submission):,} rows")
print(submission.head())
print(f"
Pred stats | mean={submission.target.mean():.6f}  std={submission.target.std():.6f}  "
      f"min={submission.target.min():.6f}  max={submission.target.max():.6f}")


## Final Summary — 10-Experiment OOF Pearson IC

In [ ]:
print("
" + "="*70)
print(f"  {'Experiment':<30s} {'OOF IC':>12s}  Note")
print("-"*70)
for name, info in results.items():
    ic_str = f"{info['ic']:.6f}" if info['ic'] is not None else "    N/A "
    print(f"  {name:<30s} {ic_str:>12s}  {info['note']}")
print("="*70)

best_exp = max((k for k in results if results[k]["ic"] is not None),
               key=lambda k: results[k]["ic"] if results[k]["ic"] is not None else -999)
print(f"
  Best experiment: {best_exp}  IC={results[best_exp]['ic']:.6f}")
